In [1]:
!pip install flask pyngrok -q

In [2]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [4]:
from google.colab import files
uploaded = files.upload()

Saving anomaly_detector.h5 to anomaly_detector.h5
Saving anomaly_threshold.pkl to anomaly_threshold.pkl
Saving cnn_lstm_classifier.h5 to cnn_lstm_classifier.h5
Saving federated_global.h5 to federated_global.h5
Saving gru_predictor.h5 to gru_predictor.h5


In [5]:
import os
for f in os.listdir('/content'):
    print(f)

.config
federated_global.h5
cnn_lstm_classifier.h5
gru_predictor.h5
anomaly_threshold.pkl
anomaly_detector.h5
sample_data


In [7]:
classifier = tf.keras.models.load_model('/content/cnn_lstm_classifier.h5', compile=False)
anomaly_model = tf.keras.models.load_model('/content/anomaly_detector.h5', compile=False)
gru_model = tf.keras.models.load_model('/content/gru_predictor.h5', compile=False)

with open('/content/anomaly_threshold.pkl', 'rb') as f:
    anomaly_threshold = pickle.load(f)

print("All models loaded!")

All models loaded!


In [8]:
from pyngrok import ngrok

# Replace "YOUR_AUTHTOKEN" with your actual ngrok authtoken
ngrok.set_auth_token("YOUR_AUTHTOKEN")
print("ngrok authtoken set. Now you can re-run the cell that connects to ngrok.")

ERROR:pyngrok.process.ngrok:t=2026-04-05T02:38:01+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T02:38:01+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T02:38:01+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [11]:
from pyngrok import ngrok
ngrok.set_auth_token("3BvBgmhkGFL2eZq8P7cKePj8gGb_6YP63oz1b2rkVMw6Gyfji")

In [43]:
import pickle
import numpy as np
import tensorflow as tf

print("Loading models...")
classifier = tf.keras.models.load_model('/content/cnn_lstm_classifier.h5', compile=False)
anomaly_model = tf.keras.models.load_model('/content/anomaly_detector.h5', compile=False)
gru_model = tf.keras.models.load_model('/content/gru_predictor.h5', compile=False)
with open('/content/anomaly_threshold.pkl', 'rb') as f:
    anomaly_threshold = pickle.load(f)

print("All models loaded!")

Loading models...
All models loaded!


In [16]:
from pyngrok import ngrok
ngrok.set_auth_token("3BvBgmhkGFL2eZq8P7cKePj8gGb_6YP63oz1b2rkVMw6Gyfji")

In [18]:
import os
print(os.listdir('/content'))

['.config', 'federated_global.h5', 'cnn_lstm_classifier.h5', 'gru_predictor.h5', 'anomaly_threshold.pkl', 'anomaly_detector.h5', 'sample_data']


In [19]:
!pip install flask pyngrok -q


Loading models...
✅ All models loaded!


In [21]:
from pyngrok import ngrok
ngrok.set_auth_token("3BvBgmhkGFL2eZq8P7cKePj8gGb_6YP63oz1b2rkVMw6Gyfji")

In [22]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        window = np.array(data['window'])
        window = window.reshape(1, window.shape[0], window.shape[1])

        cls_pred = classifier.predict(window, verbose=0)
        ano_pred = anomaly_model.predict(window, verbose=0)
        gru_pred = gru_model.predict(window, verbose=0)

        reconstruction_error = float(np.mean(np.abs(ano_pred - window)))
        is_anomaly = reconstruction_error > float(anomaly_threshold)

        icu_prob = float(gru_pred[0][0])
        cls_confidence = float(np.max(cls_pred[0]))

        if icu_prob > 0.7 or is_anomaly:
            severity = "critical"
        elif icu_prob > 0.4:
            severity = "warning"
        else:
            severity = "normal"

        return jsonify({
            "severity": severity,
            "icu_probability": icu_prob,
            "confidence": cls_confidence,
            "anomaly_score": reconstruction_error,
            "is_anomaly": is_anomaly,
            "conditions": ["Critical" if severity == "critical" else "Stable"],
            "patient_status": "Critical - Immediate Attention" if severity == "critical" else "Stable"
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

public_url = ngrok.connect(5000)
print(f"\n🚀 Public URL: {public_url}")
print("Copy this URL — you'll need it for Lambda!\n")

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()


🚀 Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5000"
Copy this URL — you'll need it for Lambda!



In [23]:
print(type(anomaly_threshold))
print(anomaly_threshold)

<class 'dict'>
{'threshold': 1.0294954776763916, 'percentile': 95.0}


In [24]:
# Extract the actual float value
threshold_value = anomaly_threshold['threshold']
print("Threshold:", threshold_value)

Threshold: 1.0294954776763916


In [25]:
from flask import Flask, request, jsonify
import threading

# Kill existing flask if running
import os, signal
app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        window = np.array(data['window'])
        window = window.reshape(1, window.shape[0], window.shape[1])

        cls_pred = classifier.predict(window, verbose=0)
        ano_pred = anomaly_model.predict(window, verbose=0)
        gru_pred = gru_model.predict(window, verbose=0)

        reconstruction_error = float(np.mean(np.abs(ano_pred - window)))
        is_anomaly = reconstruction_error > threshold_value

        icu_prob = float(gru_pred[0][0])
        cls_confidence = float(np.max(cls_pred[0]))

        if icu_prob > 0.7 or is_anomaly:
            severity = "critical"
        elif icu_prob > 0.4:
            severity = "warning"
        else:
            severity = "normal"

        return jsonify({
            "severity": severity,
            "icu_probability": icu_prob,
            "confidence": cls_confidence,
            "anomaly_score": reconstruction_error,
            "is_anomaly": is_anomaly,
            "conditions": ["Critical" if severity == "critical" else "Stable"],
            "patient_status": "Critical - Immediate Attention" if severity == "critical" else "Stable"
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

public_url = ngrok.connect(5000)
print(f"\n🚀 Public URL: {public_url}")

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 05:06:13] "POST /predict HTTP/1.1" 500 -



🚀 Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 05:06:13] "POST /predict HTTP/1.1" 500 -
Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [26]:
import threading
from flask import Flask, request, jsonify

app2 = Flask(__name__)

@app2.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        window = np.array(data['window'])
        window = window.reshape(1, window.shape[0], window.shape[1])

        cls_pred = classifier.predict(window, verbose=0)
        ano_pred = anomaly_model.predict(window, verbose=0)
        gru_pred = gru_model.predict(window, verbose=0)

        reconstruction_error = float(np.mean(np.abs(ano_pred - window)))
        is_anomaly = reconstruction_error > threshold_value

        icu_prob = float(gru_pred[0][0])
        cls_confidence = float(np.max(cls_pred[0]))

        if icu_prob > 0.7 or is_anomaly:
            severity = "critical"
        elif icu_prob > 0.4:
            severity = "warning"
        else:
            severity = "normal"

        return jsonify({
            "severity": severity,
            "icu_probability": icu_prob,
            "confidence": cls_confidence,
            "anomaly_score": reconstruction_error,
            "is_anomaly": is_anomaly,
            "conditions": ["Critical" if severity == "critical" else "Stable"],
            "patient_status": "Critical - Immediate Attention" if severity == "critical" else "Stable"
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app2.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

# New tunnel on port 5001
ngrok.disconnect("http://localhost:5000")
public_url2 = ngrok.connect(5001)
print(f"\n🚀 New URL: {public_url2}")

threading.Thread(target=lambda: app2.run(port=5001, use_reloader=False)).start()


🚀 New URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5001"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit


In [27]:
print(classifier.output_shape)

(None, 6)


In [28]:
import json
with open('/content/sample_data/../training_metrics.json', 'r') as f:
    print(json.load(f))

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/../training_metrics.json'

In [1]:
print("Classifier output shape:", classifier.output_shape)
print("GRU output shape:", gru_model.output_shape)
print("Anomaly output shape:", anomaly_model.output_shape)

# Check what the classifier actually outputs on a test input
test_window = np.random.rand(1, 200, 5)
cls_out = classifier.predict(test_window, verbose=0)
gru_out = gru_model.predict(test_window, verbose=0)
print("\nClassifier output:", cls_out)
print("GRU output:", gru_out)

NameError: name 'classifier' is not defined

In [2]:
import pickle
import numpy as np
import tensorflow as tf

classifier = tf.keras.models.load_model('/content/cnn_lstm_classifier.h5', compile=False)
anomaly_model = tf.keras.models.load_model('/content/anomaly_detector.h5', compile=False)
gru_model = tf.keras.models.load_model('/content/gru_predictor.h5', compile=False)

with open('/content/anomaly_threshold.pkl', 'rb') as f:
    anomaly_threshold = pickle.load(f)
threshold_value = anomaly_threshold['threshold']

print("✅ All models loaded!")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/content/cnn_lstm_classifier.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [4]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [5]:
import shutil
import os

# Create a folder in Drive for your models
os.makedirs('/drive/MyDrive/vitallink_models', exist_ok=True)

# Copy all model files there
for f in ['cnn_lstm_classifier.h5', 'anomaly_detector.h5', 'gru_predictor.h5', 'anomaly_threshold.pkl', 'federated_global.h5']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'/drive/MyDrive/vitallink_models/{f}')
        print(f"✅ Copied {f}")
    else:
        print(f"❌ Missing {f} — need to re-upload")

❌ Missing cnn_lstm_classifier.h5 — need to re-upload
❌ Missing anomaly_detector.h5 — need to re-upload
❌ Missing gru_predictor.h5 — need to re-upload
❌ Missing anomaly_threshold.pkl — need to re-upload
❌ Missing federated_global.h5 — need to re-upload


In [7]:
from google.colab import files
import shutil
import os

os.makedirs('/drive/MyDrive/vitallink_models', exist_ok=True)
uploaded = files.upload()

# Save directly to Drive
for filename in uploaded.keys():
    shutil.copy(f'/content/{filename}', f'/drive/MyDrive/vitallink_models/{filename}')
    print(f"✅ {filename} saved to Drive!")

Saving anomaly_detector.h5 to anomaly_detector (1).h5
Saving anomaly_threshold.pkl to anomaly_threshold.pkl
Saving cnn_lstm_classifier.h5 to cnn_lstm_classifier.h5
Saving federated_global.h5 to federated_global.h5
Saving gru_predictor.h5 to gru_predictor.h5
✅ anomaly_detector (1).h5 saved to Drive!
✅ anomaly_threshold.pkl saved to Drive!
✅ cnn_lstm_classifier.h5 saved to Drive!
✅ federated_global.h5 saved to Drive!
✅ gru_predictor.h5 saved to Drive!


In [8]:
import os
print(os.listdir('/drive/MyDrive/vitallink_models'))

['anomaly_detector.h5', 'anomaly_detector (1).h5', 'anomaly_threshold.pkl', 'cnn_lstm_classifier.h5', 'federated_global.h5', 'gru_predictor.h5']


In [9]:
import pickle
import numpy as np
import tensorflow as tf
import os

MODEL_DIR = '/drive/MyDrive/vitallink_models'

print("Loading models...")
classifier = tf.keras.models.load_model(f'{MODEL_DIR}/cnn_lstm_classifier.h5', compile=False)
anomaly_model = tf.keras.models.load_model(f'{MODEL_DIR}/anomaly_detector.h5', compile=False)
gru_model = tf.keras.models.load_model(f'{MODEL_DIR}/gru_predictor.h5', compile=False)

with open(f'{MODEL_DIR}/anomaly_threshold.pkl', 'rb') as f:
    anomaly_threshold = pickle.load(f)
threshold_value = anomaly_threshold['threshold']

print("✅ All models loaded from Drive!")

Loading models...
✅ All models loaded from Drive!


In [10]:
test_window = np.random.rand(1, 200, 5)
cls_out = classifier.predict(test_window, verbose=0)
gru_out = gru_model.predict(test_window, verbose=0)
print("Classifier output:", cls_out)
print("Classifier shape:", cls_out.shape)
print("GRU output:", gru_out)
print("GRU shape:", gru_out.shape)

Classifier output: [[9.9930251e-01 3.2145743e-07 1.2097243e-04 2.2831016e-07 6.6159184e-05
  4.0400378e-14]]
Classifier shape: (1, 6)
GRU output: [[6.0917291e-06 7.0988705e-07 7.1044466e-05]]
GRU shape: (1, 3)


In [11]:
CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])  # shape: (200, num_features)
    window = window.reshape(1, 200, -1)

    # Run both models
    condition_probs = classifier.predict(window)[0]
    severity_probs = gru_model.predict(window)[0]

    condition_idx = int(np.argmax(condition_probs))
    severity_idx = int(np.argmax(severity_probs))

    condition = CONDITION_CLASSES[condition_idx]
    severity = SEVERITY_CLASSES[severity_idx]
    confidence = float(np.max(condition_probs))

    return jsonify({
        "condition": condition,
        "severity": severity,
        "confidence": round(confidence * 100, 2),
        "condition_idx": condition_idx,
        "severity_idx": severity_idx
    })

NameError: name 'app' is not defined

In [12]:
from flask import Flask, request, jsonify
import numpy as np

app = Flask(__name__)  # 👈 this was missing

CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])
    window = window.reshape(1, 200, -1)

    condition_probs = classifier.predict(window)[0]
    severity_probs = gru_model.predict(window)[0]

    condition_idx = int(np.argmax(condition_probs))
    severity_idx = int(np.argmax(severity_probs))

    condition = CONDITION_CLASSES[condition_idx]
    severity = SEVERITY_CLASSES[severity_idx]
    confidence = float(np.max(condition_probs))

    return jsonify({
        "condition": condition,
        "severity": severity,
        "confidence": round(confidence * 100, 2),
        "condition_idx": condition_idx,
        "severity_idx": severity_idx
    })

In [15]:
from pyngrok import ngrok
import threading

# Kill any existing ngrok tunnels first
ngrok.kill()

public_url = ngrok.connect(5000)
print("Colab public URL:", public_url)

# Run Flask in background thread
threading.Thread(target=lambda: app.run(port=5000)).start()

ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:33+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:33+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:33+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [14]:
!pip install pyngrok -q

In [17]:
# Alternative: use ngrok binary directly
!pip install pyngrok -q
from pyngrok import ngrok, conf

# If you have an ngrok authtoken (recommended — avoids session limits)
# Get yours free at https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3Bv8iiqJTewWpPkzPfrw7NZXyz4_VykeKizsFwQiHoCN1Pnr")

from pyngrok import ngrok
import threading

ngrok.kill()
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5000"


In [18]:
!pip install pyngrok -q

In [19]:
from flask import Flask, request, jsonify
import numpy as np

app = Flask(__name__)

CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])
    window = window.reshape(1, 200, -1)

    condition_probs = classifier.predict(window)[0]
    severity_probs = gru_model.predict(window)[0]

    condition_idx = int(np.argmax(condition_probs))
    severity_idx = int(np.argmax(severity_probs))

    return jsonify({
        "condition": CONDITION_CLASSES[condition_idx],
        "severity": SEVERITY_CLASSES[severity_idx],
        "confidence": round(float(np.max(condition_probs)) * 100, 2)
    })

In [21]:
from pyngrok import ngrok
import threading

ngrok.set_auth_token("3Bv8iiqJTewWpPkzPfrw7NZXyz4_VykeKizsFwQiHoCN1Pnr")  # 👈 paste your token
ngrok.kill()

public_url = ngrok.connect(5000)
print("✅ Public URL:", public_url)

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

✅ Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [22]:
from pyngrok import ngrok
import threading

# Kill existing ngrok tunnels
ngrok.kill()

# Use port 5001 instead
public_url = ngrok.connect(5001)
print("✅ Public URL:", public_url)

threading.Thread(target=lambda: app.run(port=5001, use_reloader=False)).start()


✅ Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5001"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001


In [23]:
import requests
import numpy as np

# Fake window with 200 timesteps, 6 features
test_window = np.random.rand(200, 6).tolist()

response = requests.post(
    "https://nonimitable-beatriz-editorially.ngrok-free.dev/predict",
    json={"window": test_window}
)

print(response.json())

ERROR:__main__:Exception on /predict [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2831/3069170250.py", line 27, in predict
    condition_probs = classifier.predict(window)[0]
             

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [24]:
print("Classifier input shape:", classifier.input_shape)
print("GRU input shape:", gru_model.input_shape)

Classifier input shape: (None, 200, 5)
GRU input shape: (None, 200, 5)


In [26]:
# Check what features the model was trained on
# Look at your training data columns
print(X_train.shape)  # should be (samples, 200, 5)

NameError: name 'X_train' is not defined

In [27]:
from flask import Flask, request, jsonify
import numpy as np

app = Flask(__name__)

CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])
    window = window.reshape(1, 200, 5)  # ✅ fixed to 5 features

    condition_probs = classifier.predict(window)

In [28]:
import requests
import numpy as np

test_window = np.random.rand(200, 5).tolist()  # ✅ 5 features

response = requests.post(
    "https://nonimitable-beatriz-editorially.ngrok-free.dev/predict",
    json={"window": test_window}
)

print(response.json())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step


INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 10:14:03] "POST /predict HTTP/1.1" 200 -


{'condition': 'Normal', 'confidence': 99.87, 'severity': 'Critical'}


In [29]:
# Try to find feature info saved with the model
import os
import numpy as np

# Check if there's a scaler or feature list saved in your drive
for f in os.listdir('/content/drive/MyDrive/vitallink_models/'):
    print(f)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vitallink_models/'

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
import os
for f in os.listdir('/content/drive/MyDrive/vitallink_models/'):
    print(f)

anomaly_detector.h5
anomaly_detector (1).h5
anomaly_threshold.pkl
cnn_lstm_classifier.h5
federated_global.h5
gru_predictor.h5


In [32]:
import pickle

with open('/content/drive/MyDrive/vitallink_models/anomaly_threshold.pkl', 'rb') as f:
    threshold = pickle.load(f)

print(type(threshold))
print(threshold)

<class 'dict'>
{'threshold': 1.0294954776763916, 'percentile': 95.0}


In [33]:
from tensorflow import keras

anomaly_model = keras.models.load_model('/content/drive/MyDrive/vitallink_models/anomaly_detector.h5')
print("Anomaly detector input shape:", anomaly_model.input_shape)

ValueError: Could not deserialize 'keras.metrics.mse' because it is not a KerasSaveable subclass

In [34]:
from flask import Flask, request, jsonify
import numpy as np

app = Flask(__name__)

CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

FEATURE_ORDER = ["heart_rate", "spo2", "bp_sys", "bp_dia", "temperature"]

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])  # shape: (200, 5)
    window = window.reshape(1, 200, 5)

    condition_probs = classifier.predict(window)[0]
    severity_probs = gru_model.predict(window)[0]

    condition_idx = int(np.argmax(condition_probs))
    severity_idx = int(np.argmax(severity_probs))

    return jsonify({
        "condition": CONDITION_CLASSES[condition_idx],
        "severity": SEVERITY_CLASSES[severity_idx],
        "confidence": round(float(np.max(condition_probs)) * 100, 2)
    })

print("✅ Flask app defined")

✅ Flask app defined


In [35]:
import requests
import numpy as np

# Simulate realistic vitals like your patient (HR=125, SpO2=86, BP=119/68, Temp=37.4)
window = []
for _ in range(200):
    window.append([125, 86, 119, 68, 37.4])

response = requests.post(
    "https://nonimitable-beatriz-editorially.ngrok-free.dev/predict",
    json={"window": window}
)
print(response.json())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step


INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 10:23:57] "POST /predict HTTP/1.1" 200 -


{'condition': 'Respiratory Failure', 'confidence': 99.9, 'severity': 'Critical'}


In [39]:
from flask import Flask, request, jsonify
import numpy as np

app = Flask(__name__)

CONDITION_CLASSES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock"
}

SEVERITY_CLASSES = {
    0: "Stable",
    1: "Warning",
    2: "Critical"
}

ACTIONS = {
    "Normal":              [],
    "Cardiac Event":       ["Prepare defibrillator", "12-lead ECG ready", "Cardiology alert"],
    "Respiratory Failure": ["Prepare O2 support", "Ready suction", "Ventilator on standby"],
    "Sepsis":              ["Start IV fluids", "Blood cultures ready", "Broad-spectrum antibiotics"],
    "Hypertensive Crisis": ["IV antihypertensives ready", "Neuro alert", "CT scan on standby"],
    "Hypoglycemic Shock":  ["Dextrose IV ready", "Check glucose levels", "Endocrine alert"],
}

# In your /predict route, after getting condition name:
actions = ACTIONS.get(condition, [])
return jsonify({
    "condition": condition,
    "severity": severity,
    "confidence": round(float(confidence) * 100, 2),
    "actions": actions,              # ✅ keyed by name not int
    "icu_probability": icu_prob,
})

ICU_PROBABILITY = {
    0: 0.02,
    1: 0.85,
    2: 0.78,
    3: 0.72,
    4: 0.65,
    5: 0.60,
}

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    window = np.array(data['window'])
    window = window.reshape(1, 200, 5)

    condition_probs = classifier.predict(window)[0]
    severity_probs = gru_model.predict(window)[0]

    condition_idx = int(np.argmax(condition_probs))
    severity_idx = int(np.argmax(severity_probs))
    confidence = round(float(np.max(condition_probs)) * 100, 2)

    # Scale ICU probability by severity
    base_icu = ICU_PROBABILITY[condition_idx]
    severity_multiplier = [0.3, 0.7, 1.0][severity_idx]
    icu_prob = round(base_icu * severity_multiplier, 2)

    return jsonify({
        "condition": CONDITION_CLASSES[condition_idx],
        "severity": SEVERITY_CLASSES[severity_idx],
        "confidence": confidence,
        "actions": ACTIONS[condition_idx],
        "icu_probability": icu_prob,
        "condition_idx": condition_idx,
        "severity_idx": severity_idx
    })

print("✅ Flask app defined")

NameError: name 'condition' is not defined

In [1]:
import pickle
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import tensorflow as tf

# Load models
print("Loading models...")
classifier = tf.keras.models.load_model('/content/cnn_lstm_classifier.h5', compile=False)
anomaly_model = tf.keras.models.load_model('/content/anomaly_detector.h5', compile=False)
gru_model = tf.keras.models.load_model('/content/gru_predictor.h5', compile=False)
with open('/content/anomaly_threshold.pkl', 'rb') as f:
    anomaly_threshold = pickle.load(f)

print("All models loaded!")

# Class mappings
CONDITION_NAMES = {
    0: "Normal",
    1: "Cardiac Event",
    2: "Respiratory Failure",
    3: "Sepsis",
    4: "Hypertensive Crisis",
    5: "Hypoglycemic Shock",
}

ACTIONS = {
    "Normal":              [],
    "Cardiac Event":       ["Prepare defibrillator", "12-lead ECG ready", "Cardiology alert"],
    "Respiratory Failure": ["Prepare O2 support", "Ready suction", "Ventilator on standby"],
    "Sepsis":              ["Start IV fluids", "Blood cultures ready", "Broad-spectrum antibiotics"],
    "Hypertensive Crisis": ["IV antihypertensives ready", "Neuro alert", "CT scan on standby"],
    "Hypoglycemic Shock":  ["Dextrose IV ready", "Check glucose levels", "Endocrine alert"],
}

# Flask app
app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        window = np.array(data['window'])  # shape: (200, 5)
        window = window.reshape(1, window.shape[0], window.shape[1])

        # Run all three models
        cls_pred   = classifier.predict(window, verbose=0)
        ano_pred   = anomaly_model.predict(window, verbose=0)
        gru_pred   = gru_model.predict(window, verbose=0)

        # ✅ Condition from classifier (6 classes)
        cls_class      = int(np.argmax(cls_pred[0]))
        cls_confidence = float(np.max(cls_pred[0]))
        condition      = CONDITION_NAMES.get(cls_class, "Unknown")


       # Anomaly score
        reconstruction_error = float(np.mean(np.abs(ano_pred - window)))
        is_anomaly = reconstruction_error > float(anomaly_threshold['threshold'])

        # ✅ Severity from GRU (3 classes: 0=Stable, 1=Warning, 2=Critical)
        gru_class = int(np.argmax(gru_pred[0]))
        icu_prob  = float(gru_pred[0][2])  # probability of Critical class

        SEVERITY_NAMES = {0: "Stable", 1: "Warning", 2: "Critical"}
        severity = SEVERITY_NAMES.get(gru_class, "Unknown")

        # ✅ Actions by condition name
        actions = ACTIONS.get(condition, [])

        print(f"[predict] class={cls_class} condition={condition} severity={severity} conf={cls_confidence:.3f} icu={icu_prob:.3f}")

        return jsonify({
            "condition":       condition,
            "severity":        severity,
            "confidence":      round(cls_confidence * 100, 2),
            "actions":         actions,
            "icu_probability": icu_prob,
            "anomaly_score":   reconstruction_error,
            "is_anomaly":      is_anomaly,
        })

    except Exception as e:
        print("Predict error:", e)
        return jsonify({"error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\n🚀 Public URL: {public_url}")
print("Copy this URL — you'll need it for Lambda!\n")

# Start Flask in background thread
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

Loading models...
All models loaded!

🚀 Public URL: NgrokTunnel: "https://nonimitable-beatriz-editorially.ngrok-free.dev" -> "http://localhost:5000"
Copy this URL — you'll need it for Lambda!

 * Serving Flask app '__main__'


In [ ]:
import os
os.system("kill -9 $(lsof -t -i:5000)")

In [3]:
import requests
r = requests.post("http://localhost:5000/predict", json={"window": [[75, 98, 136, 83, 37.6]] * 200})
print(r.json())

INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 16:55:43] "POST /predict HTTP/1.1" 200 -


[predict] class=2 condition=Respiratory Failure severity=Critical conf=0.995 icu=0.149
{'actions': ['Prepare O2 support', 'Ready suction', 'Ventilator on standby'], 'anomaly_score': 85.93326100069844, 'condition': 'Respiratory Failure', 'confidence': 99.53, 'icu_probability': 0.14860795438289642, 'is_anomaly': True, 'severity': 'Critical'}


In [4]:
import numpy as np

window = np.array([[75, 98, 136, 83, 37.6]] * 200).reshape(1, 200, 5)

cls_pred = classifier.predict(window, verbose=0)
ano_pred = anomaly_model.predict(window, verbose=0)
gru_pred = gru_model.predict(window, verbose=0)

print("ano_pred:", ano_pred)
print("ano_pred shape:", ano_pred.shape)
print("window shape:", window.shape)

reconstruction_error = float(np.mean(np.abs(ano_pred - window)))
print("reconstruction_error:", reconstruction_error)

ano_pred: [[[ 9.41311941e-03 -1.37135703e-02 -1.52714849e-02  9.35979001e-03
    1.66020058e-02]
  [-1.31526198e-02 -2.00522207e-02 -1.24406870e-02  2.11351924e-02
    5.21221384e-02]
  [-4.01196629e-02 -2.65210662e-02 -1.08443759e-02  2.93387417e-02
    8.58544707e-02]
  [-6.63648695e-02 -3.25997435e-02 -9.37268883e-03  3.57497558e-02
    1.11648783e-01]
  [-8.70118216e-02 -3.80306393e-02 -6.00211322e-03  4.21771705e-02
    1.27427071e-01]
  [-9.90808010e-02 -4.34486680e-02  8.81746411e-04  5.04116490e-02
    1.33449331e-01]
  [-1.02212153e-01 -5.01487665e-02  1.13441199e-02  6.17639534e-02
    1.30949810e-01]
  [-9.78381410e-02 -5.90785295e-02  2.39264108e-02  7.64583871e-02
    1.21508598e-01]
  [-8.77977088e-02 -6.99870884e-02  3.66001911e-02  9.34868976e-02
    1.06883526e-01]
  [-7.36910701e-02 -8.14664364e-02  4.76471297e-02  1.11054264e-01
    8.89013559e-02]
  [-5.69913164e-02 -9.16176587e-02  5.59174530e-02  1.27280131e-01
    6.92710727e-02]
  [-3.92191634e-02 -9.86986756e-0

In [5]:
print("anomaly_threshold value:", anomaly_threshold)
print("anomaly_threshold type:", type(anomaly_threshold))

anomaly_threshold value: {'threshold': 1.0294954776763916, 'percentile': 95.0}
anomaly_threshold type: <class 'dict'>
